# Analyse de manuscrits médiévaux avec YOLO-gen

**Référence :** Torres Aguilar, S. (2025). *From Codicology to Code: A Comparative Study of Transformer and YOLO-based Detectors for Layout Analysis in Historical Documents*. arXiv:2506.20326  
**Modèle :** `magistermilitum/YOLO_manuscripts` (Hugging Face, licence MIT)

## 1. Installation des dépendances

In [52]:
#!pip install ultralytics huggingface_hub pillow matplotlib ipykernel

## 2. Imports et constantes

In [ ]:
import os
import sys
import json
import tempfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon, Patch
from PIL import Image
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

IMAGE_PATH  = r"C:\Users\djame\Documents\GitHub\Analyse-de-manuscrits\Exercice 3\Le_Roman_de_la_Rose_[...]Guillaume_de_btv1b6000369q_8 - Copie.jpeg"
OUTPUT_DIR  = Path("yologen_output")
HF_MODEL_ID = "magistermilitum/YOLO_manuscripts"
HF_TOKEN    = os.environ.get("HF_TOKEN", None)

CONF_PER_CLASS = {
    "Text_Main":         0.08,
    "Text":              0.06,
    "Decoration":        0.03,
    "Initial":           0.03,
    "Paratext":          0.05,
    "Paratext_Marginal": 0.05,
    "default":           0.05,
}

IOU_PER_GROUP = {
    "Text_Main": 0.3,
    "Text":      0.3,
    "default":   0.5,
}

PALETTE = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45",
    "#fabed4", "#469990"
]

## 3. Fonction `load_model`

Télécharge `best.pt` depuis Hugging Face et retourne un objet `YOLO` prêt à l'inférence.

In [54]:
def load_model() -> YOLO:
    # Résolution du token : variable globale → variable d'environnement → cache HF
    token = HF_TOKEN or os.environ.get("HF_TOKEN") or None

    try:
        model_path = hf_hub_download(
            repo_id=HF_MODEL_ID,
            filename="best.pt",
            token=token
        )
    except Exception as e:
        if "403" in str(e) or "401" in str(e):
            sys.exit(
                "[ERREUR] Accès refusé (403/401). "
                "Définissez HF_TOKEN avec un token Hugging Face valide "
                "ou lancez `huggingface-cli login`."
            )
        raise

    model = YOLO(model_path)
    print(f"Modèle chargé depuis : {model_path}")
    print(f"Classes détectables  : {model.names}")
    return model

## 4. Fonction `diagnose_thresholds`

Teste plusieurs seuils décroissants et retourne les résultats au premier seuil ayant produit au moins une détection.

In [ ]:
def crop_image(img_pil: Image.Image, bottom_pct: float = 0.04) -> Image.Image:
    w, h = img_pil.size
    return img_pil.crop((0, 0, w, int(h * (1 - bottom_pct))))

def multi_scale_predict(model: YOLO, img_pil: Image.Image) -> list:
    tmp_path = Path(tempfile.gettempdir()) / "cropped_input.jpg"
    img_pil.save(str(tmp_path), quality=95)

    all_results = []
    for sz in [640, 1280, 1920, 2560]:
        print(f"  Inférence à imgsz={sz}...")
        results = model.predict(
            source=str(tmp_path),
            conf=0.01,
            iou=0.3,
            imgsz=sz,
            verbose=False,
            augment=True,
        )
        all_results.extend(results)

    return all_results

def bbox_iou(pts1: np.ndarray, pts2: np.ndarray) -> float:
    x1_min, y1_min = pts1.min(axis=0)
    x1_max, y1_max = pts1.max(axis=0)
    x2_min, y2_min = pts2.min(axis=0)
    x2_max, y2_max = pts2.max(axis=0)

    ix_min = max(x1_min, x2_min)
    iy_min = max(y1_min, y2_min)
    ix_max = min(x1_max, x2_max)
    iy_max = min(y1_max, y2_max)

    inter = max(0, ix_max - ix_min) * max(0, iy_max - iy_min)
    area1 = (x1_max - x1_min) * (y1_max - y1_min)
    area2 = (x2_max - x2_min) * (y2_max - y2_min)
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def merge_and_filter(all_results: list) -> list:
    detections = []
    for result in all_results:
        if result.obb is None:
            continue
        names = result.names
        for box in result.obb:
            cls_id   = int(box.cls[0])
            cls_name = names[cls_id]
            score    = float(box.conf[0])
            pts      = box.xyxyxyxy[0].cpu().numpy().reshape(4, 2)
            threshold = CONF_PER_CLASS.get(cls_name, CONF_PER_CLASS["default"])
            if score >= threshold:
                detections.append({"cls_id": cls_id, "cls_name": cls_name, "score": score, "pts": pts})

    if not detections:
        return detections

    final    = []
    by_class = defaultdict(list)
    for d in detections:
        by_class[d["cls_name"]].append(d)

    for cls_name, dets in by_class.items():
        dets.sort(key=lambda d: d["score"], reverse=True)
        kept = []
        for d in dets:
            group_iou = IOU_PER_GROUP.get(cls_name, IOU_PER_GROUP["default"])
            if not any(bbox_iou(d["pts"], k["pts"]) > group_iou for k in kept):
                kept.append(d)
        final.extend(kept)

    print(f"Après fusion : {len(final)} détections (avant : {len(detections)})")
    return final

## 5. Fonction `visualize`

Dessine les boîtes OBB (polygones orientés) sur l'image et sauvegarde la figure.

In [ ]:
def visualize(detections: list, img_pil: Image.Image, save_path: Path) -> None:
    fig, ax = plt.subplots(1, 1, figsize=(14, 18))
    ax.imshow(np.array(img_pil))
    ax.axis("off")
    ax.set_title("YOLO-gen OBB — multi-scale + TTA + seuils par classe", fontsize=12)

    legend_handles = {}

    for d in detections:
        pts   = d["pts"]
        color = PALETTE[d["cls_id"] % len(PALETTE)]

        ax.add_patch(mpatches.Polygon(pts, closed=True, linewidth=0,
                                      facecolor=color, alpha=0.18))
        ax.add_patch(mpatches.Polygon(pts, closed=True, linewidth=1.8,
                                      edgecolor=color, facecolor="none"))

        top_idx = pts[:, 1].argmin()
        tx, ty  = pts[top_idx]
        ax.text(tx, ty - 5, f"{d['cls_name']} {d['score']:.2f}",
                fontsize=7, color="white",
                bbox=dict(boxstyle="round,pad=0.2", fc=color, alpha=0.85, lw=0))

        if d["cls_name"] not in legend_handles:
            legend_handles[d["cls_name"]] = mpatches.Patch(color=color, label=d["cls_name"])

    if not detections:
        ax.text(0.5, 0.5, "Aucune détection", transform=ax.transAxes,
                ha="center", va="center", fontsize=16, color="red")
    else:
        ax.legend(handles=list(legend_handles.values()),
                  loc="upper right", fontsize=9, framealpha=0.85)

    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Sauvegardé : {save_path}  ({len(detections)} détections)")

## 6. Fonction `report_and_export`

Affiche un rapport console groupé par classe et exporte les détections en JSON.

In [ ]:
def report_and_export(detections: list, save_path: Path) -> None:
    stats   = defaultdict(list)
    regions = []

    for d in detections:
        stats[d["cls_name"]].append(d["score"])
        regions.append({
            "class":      d["cls_name"],
            "confidence": round(d["score"], 4),
            "polygon":    d["pts"].tolist()
        })

    print(f"\n{'Classe':<22} {'N':>4} {'Moy':>8} {'Max':>8}")
    print("-" * 46)
    for cls_name, scores in sorted(stats.items()):
        print(f"{cls_name:<22} {len(scores):>4} {np.mean(scores):>8.3f} {max(scores):>8.3f}")
    print(f"\nTotal : {len(regions)} région(s)")

    payload = {
        "model":          HF_MODEL_ID,
        "paper":          "arXiv:2506.20326",
        "conf_per_class": CONF_PER_CLASS,
        "regions":        regions
    }
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"JSON : {save_path}")

## 7. Pipeline principal `main`

In [ ]:
def main():
    OUTPUT_DIR.mkdir(exist_ok=True)
    model = load_model()

    img_orig    = Image.open(IMAGE_PATH).convert("RGB")
    img_cropped = crop_image(img_orig)
    print(f"Image : {img_orig.size} → croppée : {img_cropped.size}")

    print("\nInférence multi-scale + TTA...")
    all_results = multi_scale_predict(model, img_cropped)
    detections  = merge_and_filter(all_results)

    visualize(detections, img_cropped, OUTPUT_DIR / "yologen_detections_v3.jpg")
    report_and_export(detections, OUTPUT_DIR / "yologen_detections_v3.json")
    bootstrap_detections_v2(detections, n_iterations=10000, ci=95)

    return model, detections

model_loaded, detections_main = main()

## 8. Comparaison sur 4 tailles d'image

In [ ]:
def compare_imgsz(model: YOLO, image_path: str, sizes: list[int], conf: float = 0.05, iou: float = 0.45):
    """Lance l'inférence sur plusieurs tailles et affiche un tableau comparatif + grille de visualisation."""

    print(f"{'imgsz':>8} | {'Détections':>10} | {'Conf. moy.':>10} | Classes détectées")
    print("-" * 65)

    fig, axes = plt.subplots(1, len(sizes), figsize=(6 * len(sizes), 10))
    img_pil = Image.open(image_path).convert("RGB")

    for ax, imgsz in zip(axes, sizes):
        results = model.predict(
            source=image_path,
            conf=conf,
            iou=iou,
            imgsz=imgsz,
            verbose=False
        )

        detections = []
        class_counts = defaultdict(int)

        for r in results:
            if r.obb is None:
                continue
            for box in r.obb:
                cls_name = model.names[int(box.cls[0])]
                score    = float(box.conf[0])
                pts      = box.xyxyxyxy[0].cpu().numpy().reshape(4, 2)
                detections.append((cls_name, score, pts))
                class_counts[cls_name] += 1

        n        = len(detections)
        mean_conf = np.mean([d[1] for d in detections]) if detections else 0.0
        classes  = ", ".join(f"{k}({v})" for k, v in sorted(class_counts.items()))

        print(f"{imgsz:>8} | {n:>10} | {mean_conf:>10.3f} | {classes}")

        # Visualisation
        ax.imshow(np.array(img_pil))
        ax.axis("off")
        ax.set_title(f"imgsz={imgsz}\n{n} détection(s)", fontsize=11)

        for cls_name, score, pts in detections:
            cls_id = [k for k, v in model.names.items() if v == cls_name][0]
            color  = PALETTE[cls_id % len(PALETTE)]
            ax.add_patch(Polygon(pts, closed=True, linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.15))
            ax.add_patch(Polygon(pts, closed=True, linewidth=1.5, edgecolor=color, facecolor="none"))
            top_idx = pts[:, 1].argmin()
            ax.text(pts[top_idx][0], pts[top_idx][1] - 4,
                    f"{cls_name} {score:.2f}", fontsize=6, color="white",
                    bbox=dict(facecolor=color, alpha=0.8, linewidth=0, boxstyle="round,pad=0.2"))

    plt.suptitle(f"Comparaison imgsz — conf={conf}, iou={iou}", fontsize=13, y=1.01)
    plt.tight_layout()
    out_path = OUTPUT_DIR / "compare_imgsz.jpg"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\nFigure sauvegardée : {out_path}")


# Lancement sur 4 tailles
model_loaded = load_model()
OUTPUT_DIR.mkdir(exist_ok=True)
compare_imgsz(model_loaded, IMAGE_PATH, sizes=[640, 1080, 1280, 1920], conf=0.05)

## 9. Bootstrap statistique sur les détections

In [ ]:
def bootstrap_detections_v2(detections: list, n_iterations: int = 10000, ci: float = 95.0):
    """Bootstrap adapté au nouveau format list[dict] avec cls_name et score."""
    if not detections:
        print("Aucune détection à bootstrapper.")
        return

    classes = sorted(set(d["cls_name"] for d in detections))
    alpha   = (100 - ci) / 2

    print(f"Bootstrap : {n_iterations} itérations sur {len(detections)} détections\n")
    print(f"{'Classe':<25} {'Moy. obs.':>10} {'IC bas':>10} {'IC haut':>10} {'n':>5}")
    print("-" * 65)

    fig, axes = plt.subplots(1, len(classes), figsize=(4 * len(classes), 4), squeeze=False)

    for ax, cls_name in zip(axes[0], classes):
        cls_scores = np.array([d["score"] for d in detections if d["cls_name"] == cls_name])
        observed   = np.mean(cls_scores)

        boot_means = np.array([
            np.mean(np.random.choice(cls_scores, size=len(cls_scores), replace=True))
            for _ in range(n_iterations)
        ])

        lo = np.percentile(boot_means, alpha)
        hi = np.percentile(boot_means, 100 - alpha)

        print(f"{cls_name:<25} {observed:>10.4f} {lo:>10.4f} {hi:>10.4f} {len(cls_scores):>5}")

        ax.hist(boot_means, bins=60, color="#4363d8", alpha=0.7, edgecolor="none")
        ax.axvline(observed, color="red",    linestyle="--", linewidth=1.5, label=f"obs={observed:.3f}")
        ax.axvline(lo,       color="orange", linestyle=":",  linewidth=1.2, label=f"IC {ci:.0f}%")
        ax.axvline(hi,       color="orange", linestyle=":",  linewidth=1.2)
        ax.set_title(cls_name, fontsize=9)
        ax.set_xlabel("Conf. moy.", fontsize=8)
        ax.legend(fontsize=7)

    plt.suptitle(f"Distributions bootstrap ({n_iterations} itérations)", fontsize=12)
    plt.tight_layout()
    out_path = OUTPUT_DIR / "bootstrap.jpg"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Figure sauvegardée : {out_path}")